In [2]:
import shutil
import os

# Folder path to delete
folder_path = r"/mnt/lustre/lalit47/down/seg/DDTI"

# Check if folder exists
if os.path.exists(folder_path):

    # Delete folder and all contents
    shutil.rmtree(folder_path)

    print(f"Deleted folder: {folder_path}")

else:
    print("Folder does not exist")

Deleted folder: /mnt/lustre/lalit47/down/seg/DDTI


In [ ]:
import os
import cv2
import random
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/mnt/lustre/lalit47/us_data/bench/DDTI/1_or_data"
target_path = r"//mnt/lustre/lalit47/down/seg/DDTI"

images_folder = os.path.join(source_path, "image")
masks_folder = os.path.join(source_path, "mask")

# Split ratio: train : val : test = 7 : 1 : 2
train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)


def process_and_save(src, dst):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG
    """

    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)

    return True


def get_class_id(filename):
    """
    Extract class id from filename.

    Modify this function according to your filename pattern.

    Example:
    image name: 001_3.JPG
    class id: 3
    """

    base_name = os.path.splitext(filename)[0]

    # Example: 001_3.JPG -> class_id = 3
    class_id = base_name.split("_")[-1]

    return class_id


def collect_image_mask_pairs():
    """
    Collect images and corresponding masks.
    Image and mask should have same base name.
    """

    pairs_by_class = defaultdict(list)

    image_files = [
        f for f in os.listdir(images_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for image_file in image_files:
        base_name = os.path.splitext(image_file)[0]

        image_path = os.path.join(images_folder, image_file)

        # Mask can be .png, .jpg, or .jpeg
        mask_path = None
        for ext in [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]:
            possible_mask = os.path.join(masks_folder, base_name + ext)
            if os.path.exists(possible_mask):
                mask_path = possible_mask
                break

        if mask_path is None:
            print(f"Mask missing for: {image_file}")
            continue

        class_id = get_class_id(image_file)

        pairs_by_class[class_id].append((image_path, mask_path, base_name))

    return pairs_by_class


def split_stratified(pairs_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_id, pairs in pairs_by_class.items():
        random.shuffle(pairs)

        n = len(pairs)

        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_pairs = pairs[:n_train]
        val_pairs = pairs[n_train:n_train + n_val]
        test_pairs = pairs[n_train + n_val:]

        train_data.extend(train_pairs)
        val_data.extend(val_pairs)
        test_data.extend(test_pairs)

        print(
            f"Class {class_id}: "
            f"total={n}, train={len(train_pairs)}, "
            f"val={len(val_pairs)}, test={len(test_pairs)}"
        )

    return train_data, val_data, test_data


def save_split(data, split_name):
    image_out_dir = os.path.join(target_path, split_name, "images")
    mask_out_dir = os.path.join(target_path, split_name, "masks")

    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    for image_path, mask_path, base_name in data:
        image_dst = os.path.join(image_out_dir, base_name + ".png")
        mask_dst = os.path.join(mask_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)
        process_and_save(mask_path, mask_dst)


# =========================
# MAIN
# =========================

pairs_by_class = collect_image_mask_pairs()

train_data, val_data, test_data = split_stratified(pairs_by_class)

save_split(train_data, "train")
save_split(val_data, "val")
save_split(test_data, "test")

print("\nStratified dataset split completed successfully!")

In [3]:
import os
import cv2
import random
import pandas as pd
from collections import defaultdict

# =========================
# USER INPUT PATHS
# =========================

ource_path = r"/mnt/lustre/lalit47/us_data/bench/DDTI/1_or_data"
target_path = r"//mnt/lustre/lalit47/down/seg/DDTI"

images_folder = os.path.join(source_path, "image")
masks_folder = os.path.join(source_path, "mask")

csv_file = os.path.join(source_path, "category.csv")  # CSV with ID,CATE

train_ratio = 0.7
val_ratio = 0.1
test_ratio = 0.2

random_seed = 42
random.seed(random_seed)


def process_and_save(src, dst):
    """
    Read image in grayscale,
    resize to 256x256,
    save as PNG
    """
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)
    return True


def collect_image_mask_pairs_from_csv():
    """
    CSV format:
    ID,CATE
    1.PNG,0
    10.PNG,0
    100.PNG,1
    """

    df = pd.read_csv(csv_file)

    pairs_by_class = defaultdict(list)

    for _, row in df.iterrows():
        image_name = str(row["ID"])
        class_id = str(row["CATE"])

        base_name = os.path.splitext(image_name)[0]

        image_path = os.path.join(images_folder, image_name)

        # Find mask with same base name
        mask_path = None
        for ext in [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]:
            possible_mask = os.path.join(masks_folder, base_name + ext)
            if os.path.exists(possible_mask):
                mask_path = possible_mask
                break

        if not os.path.exists(image_path):
            print(f"Image missing: {image_path}")
            continue

        if mask_path is None:
            print(f"Mask missing for: {image_name}")
            continue

        pairs_by_class[class_id].append((image_path, mask_path, base_name))

    return pairs_by_class


def split_stratified(pairs_by_class):
    train_data = []
    val_data = []
    test_data = []

    for class_id, pairs in pairs_by_class.items():
        random.shuffle(pairs)

        n = len(pairs)

        n_train = int(n * train_ratio)
        n_val = int(n * val_ratio)

        train_pairs = pairs[:n_train]
        val_pairs = pairs[n_train:n_train + n_val]
        test_pairs = pairs[n_train + n_val:]

        train_data.extend(train_pairs)
        val_data.extend(val_pairs)
        test_data.extend(test_pairs)

        # print(
        #     f"Class {class_id}: total={n}, "
        #     f"train={len(train_pairs)}, "
        #     f"val={len(val_pairs)}, "
        #     f"test={len(test_pairs)}"
        # )

    return train_data, val_data, test_data


def save_split(data, split_name):
    image_out_dir = os.path.join(target_path, split_name, "images")
    mask_out_dir = os.path.join(target_path, split_name, "masks")

    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    for image_path, mask_path, base_name in data:
        image_dst = os.path.join(image_out_dir, base_name + ".png")
        mask_dst = os.path.join(mask_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)
        process_and_save(mask_path, mask_dst)


# =========================
# MAIN
# =========================

pairs_by_class = collect_image_mask_pairs_from_csv()

train_data, val_data, test_data = split_stratified(pairs_by_class)

save_split(train_data, "train")
save_split(val_data, "val")
save_split(test_data, "test")

print("\nStratified train/val/test split completed successfully!")


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/ipyk

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/mnt/lustre2/shared_conda/envs/lalit47/env_pytorch/lib/python3.11/site-packages/ipyk

AttributeError: _ARRAY_API not found


Stratified train/val/test split completed successfully!


In [ ]:
import os
import cv2
import random

# =========================
# USER INPUT PATHS
# =========================

source_path = r"/path/to/source_dataset"
target_path = r"/path/to/output_dataset"

images_folder = os.path.join(source_path, "images")
masks_folder = os.path.join(source_path, "masks")

random_seed = 42
random.seed(random_seed)


def process_and_save(src, dst):
    im = cv2.imread(src, cv2.IMREAD_GRAYSCALE)

    if im is None:
        print(f"Failed to read: {src}")
        return False

    im = cv2.resize(im, (256, 256), interpolation=cv2.INTER_LINEAR)
    cv2.imwrite(dst, im)

    return True


def collect_image_mask_pairs():
    pairs = []

    image_files = [
        f for f in os.listdir(images_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    for image_file in image_files:
        base_name = os.path.splitext(image_file)[0]
        image_path = os.path.join(images_folder, image_file)

        mask_path = None
        for ext in [".png", ".PNG", ".jpg", ".JPG", ".jpeg", ".JPEG"]:
            possible_mask = os.path.join(masks_folder, base_name + ext)

            if os.path.exists(possible_mask):
                mask_path = possible_mask
                break

        if mask_path is None:
            print(f"Mask missing for: {image_file}")
            continue

        pairs.append((image_path, mask_path, base_name))

    return pairs


def split_random(pairs):
    random.shuffle(pairs)

    n = len(pairs)

    n_train = int(n * 0.7)
    n_val = int(n * 0.1)

    train_data = pairs[:n_train]
    val_data = pairs[n_train:n_train + n_val]
    test_data = pairs[n_train + n_val:]

    print(f"Total images: {n}")
    print(f"Train: {len(train_data)}")
    print(f"Val: {len(val_data)}")
    print(f"Test: {len(test_data)}")

    return train_data, val_data, test_data


def save_split(data, split_name):
    image_out_dir = os.path.join(target_path, split_name, "images")
    mask_out_dir = os.path.join(target_path, split_name, "masks")

    os.makedirs(image_out_dir, exist_ok=True)
    os.makedirs(mask_out_dir, exist_ok=True)

    for image_path, mask_path, base_name in data:
        image_dst = os.path.join(image_out_dir, base_name + ".png")
        mask_dst = os.path.join(mask_out_dir, base_name + ".png")

        process_and_save(image_path, image_dst)
        process_and_save(mask_path, mask_dst)


pairs = collect_image_mask_pairs()

train_data, val_data, test_data = split_random(pairs)

save_split(train_data, "train")
save_split(val_data, "val")
save_split(test_data, "test")

print("\nRandom dataset split completed successfully!")